# Algorithmic Systems Design: Smart Spell-Checker & Autocorrect Pipeline
**Authors:** Jakub Habib, Ulugbek Tojiboev

### System Objective
A real-time pipeline that verifies user text input and instantly suggests the top 3 statistically probable corrections for typos. The system chains three distinct algorithms to process data efficiently.

In [3]:
import re

class Vocabulary:
    def __init__(self):
        # Maps word (str) to its frequency (int)
        self.word_frequencies = {}

    def clean_text(self, text):
        """Normalize data: remove non-alphabetic characters and convert to lowercase."""
        if not text:
            return ""
        cleaned = re.sub(r'[^a-zA-Z]', '', text)
        return cleaned.lower()

    def load_dictionary(self, raw_data):
        """Build the Hash Map from a list of (word, frequency) tuples."""
        for word, freq in raw_data:
            cleaned_word = self.clean_text(word)
            if cleaned_word:
                self.word_frequencies[cleaned_word] = freq
        print(f"Loaded {len(self.word_frequencies)} words into the dictionary.")

    def is_valid_word(self, word):
        """Fast O(1) validation. Checks if the word exists in the dictionary."""
        cleaned_word = self.clean_text(word)
        if not cleaned_word:
            return True # Ignore empty strings

        # O(1) Hash Map lookup
        return cleaned_word in self.word_frequencies

# --- MOCK TEST FOR STAGE 1 ---
if __name__ == "__main__":
    mock_database = [
        ("the", 100000),
        ("tea", 500),
        ("ten", 3000),
        ("hello", 50000),
        ("definitely", 20000)
    ]

    vocab = Vocabulary()
    vocab.load_dictionary(mock_database)

    test_word_1 = "Hello!"      
    test_word_2 = "definetly"   

    print(f"Is 'Hello!' valid? {vocab.is_valid_word(test_word_1)}")
    print(f"Is 'definetly' valid? {vocab.is_valid_word(test_word_2)}")

Loaded 5 words into the dictionary.
Is 'Hello!' valid? True
Is 'definetly' valid? False


In [2]:
import heapq

class TopKRanker:
    def __init__(self, k=3):
        self.k = k

    def get_top_k(self, candidates, word_frequencies):
        min_heap = []
        
        for word in candidates:
            # Default to 0 if word is somehow missing from vocabulary
            freq = word_frequencies.get(word, 0) 
            
            # Push tuple (frequency, word) into the min-heap
            heapq.heappush(min_heap, (freq, word))
            
            # Maintain strict heap size of k to achieve O(C log k) time
            if len(min_heap) > self.k:
                heapq.heappop(min_heap)
        
        # Sort the remaining k elements in descending order (O(k log k))
        result = sorted(min_heap, key=lambda x: x[0], reverse=True)
        
        # Extract just the words
        return [word for freq, word in result]

# --- MOCK TEST FOR STAGE 3 ---
if __name__ == "__main__":
    # Mock data representing the Hash Map from Stage 1
    mock_frequencies = {
        "tea": 500,
        "ten": 3000,
        "the": 100000,
        "ted": 150,
        "tech": 15000
    }
    
    # Mock candidates representing the output from Stage 2 (Levenshtein)
    # Simulated typo: "teh"
    mock_candidates = ["tea", "ten", "the", "ted", "tech"]
    
    ranker = TopKRanker(k=3)
    top_suggestions = ranker.get_top_k(mock_candidates, mock_frequencies)
    
    print(f"Raw Candidates: {mock_candidates}")
    print(f"Top 3 Suggestions: {top_suggestions}")

Raw Candidates: ['tea', 'ten', 'the', 'ted', 'tech']
Top 3 Suggestions: ['the', 'tech', 'ten']
